In [1]:
import pandas as pd
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pickle
import json

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
GPU name: NVIDIA GeForce GTX 1080 Ti


- Pada bagian ini, dataset hasil preprocessing (X dan y) akan dibagi menjadi beberapa fold menggunakan K-Fold.  
- Setiap fold akan dilatih dan dievaluasi menggunakan LSTM dan GRU.  
- Tujuannya: memperoleh performa rata-rata yang stabil dan memilih model terbaik.

### Step 1 — Load artefak preprocessing
Kita ambil vocab, label encoder, config, dan dataset encoded yang dibuat di fase 1.

In [2]:
# Load vocab, label encoder, config
with open("artifacts/vocab/word2idx_aug.pkl", "rb") as f:
    word2idx = pickle.load(f)

with open("artifacts/labels/label_encoder_aug.pkl", "rb") as f:
    label_encoder = pickle.load(f)

with open("artifacts/config/config_aug.json") as f:
    config = json.load(f)

MAX_LEN = config["max_len"]
VOCAB_SIZE = config["vocab_size"]
NUM_CLASSES = config["num_classes"]
EMBED_DIM = config["embedding_dim"]
HIDDEN_SIZE = config["hidden_size"]

# Load balanced & encoded dataset
X = np.load("artifacts/dataset/X_aug.npy")
y = np.load("artifacts/dataset/y_aug.npy")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Step 2 — Siapkan Dataset dan DataLoader
Dataset ini dipake PyTorch buat ngasih data dalam bentuk batch.


In [3]:
class NewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = NewsDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

### Step 3 — Bikin arsitektur model LSTM dan GRU
Kedua model ini mirip, cuma beda di jenis recurrent layer-nya.  
Tambahin Word Embedding juga

In [4]:
embedding_matrix = np.load("artifacts/embedding/embedding_matrix_aug.npy")
EMBED_DIM = embedding_matrix.shape[1]

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False   # kalau mau fine-tuning, True kalau mau tetap
        )
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False   # kalau mau fine-tuning, True kalau mau tetap
        )
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.emb(x)
        _, h = self.gru(x)
        return self.fc(h[-1])

### Step 4 — Setup K-Fold untuk bagi dataset

In [6]:
k = 5
kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

In [7]:
for train_idx, test_idx in kf.split(X, y):
    print(len(train_idx), len(test_idx))

70156 17540
70157 17539
70157 17539
70157 17539
70157 17539


### Step 5 — Loop K-Fold Training + Evaluasi

In [8]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def encode_text(text):
    tokens = clean_text(text).split()
    seq = [word2idx.get(tok, word2idx["<UNK>"]) for tok in tokens]

    if len(seq) < MAX_LEN:
        seq = seq + [word2idx["<PAD>"]] * (MAX_LEN - len(seq))
    else:
        seq = seq[:MAX_LEN]

    return torch.tensor([seq], dtype=torch.long).to(device)

In [9]:
lstm_model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_CLASSES).to(device)
print(lstm_model.emb.weight[10][:10])

test = encode_text("jokowi resmikan proyek tol baru")

# pastikan list int
if isinstance(test, torch.Tensor):
    test = test.tolist()

inp = torch.tensor([test], dtype=torch.long).to(device)

vecs = lstm_model.emb(inp).shape
vecs

tensor([-0.7456, -0.1253, -1.0992,  0.1143,  0.0916,  0.1585, -0.1856, -0.0210,
         0.5227, -0.2509], device='cuda:0', grad_fn=<SliceBackward0>)


torch.Size([1, 1, 20, 300])

In [10]:
EPOCHS = 5
BATCH_SIZE = 32
lstm_scores = []
gru_scores = []
fold_no = 1

for train_idx, test_idx in kf.split(X, y):
    print(f"\n===== FOLD {fold_no} =====")

    # Split data
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Dataset & Loader
    train_loader = DataLoader(
        NewsDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True
    )
    test_loader = DataLoader(
        NewsDataset(X_test, y_test), batch_size=BATCH_SIZE
    )

    # Init models
    lstm_model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_CLASSES).to(device)
    gru_model  = GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_CLASSES).to(device)

    criterion = nn.CrossEntropyLoss()
    lstm_opt = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
    gru_opt  = torch.optim.Adam(gru_model.parameters(), lr=1e-3)

    # === TRAIN LSTM ===
    for epoch in range(EPOCHS):
        lstm_model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            lstm_opt.zero_grad()
            out = lstm_model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            lstm_opt.step()

    # === TRAIN GRU ===
    for epoch in range(EPOCHS):
        gru_model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            gru_opt.zero_grad()
            out = gru_model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            gru_opt.step()

    # === EVALUASI ===
    def evaluate(model):

        model.eval()
        preds = []
        true = []

        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                out = model(Xb)
                pred = torch.argmax(out, dim=1).cpu().tolist()
                true.extend(yb.tolist())
                preds.extend(pred)

        acc = accuracy_score(true, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(
            true, preds, average="macro"
        )

        return acc, prec, rec, f1

    # simpan skor
    lstm_scores.append(evaluate(lstm_model))
    gru_scores.append(evaluate(gru_model))

    print(f"LSTM fold {fold_no}:", lstm_scores[-1])
    print(f"GRU  fold {fold_no}:", gru_scores[-1])

    fold_no += 1



===== FOLD 1 =====
LSTM fold 1: (0.9283352337514253, 0.9290823222321899, 0.9283340515099445, 0.928382993550401)
GRU  fold 1: (0.927023945267959, 0.9273868153563968, 0.9270220476122414, 0.926991253013976)

===== FOLD 2 =====
LSTM fold 2: (0.9260505159929301, 0.9263515720194797, 0.9260492899767713, 0.9260699512969057)
GRU  fold 2: (0.9265066423399282, 0.9272023551204985, 0.92650747143653, 0.9265038479617458)

===== FOLD 3 =====
LSTM fold 3: (0.9270768002736758, 0.9280495687312391, 0.927080900634442, 0.9271509910425149)
GRU  fold 3: (0.9277039740007982, 0.9279002110542736, 0.9277065688401254, 0.9275983653403725)

===== FOLD 4 =====
LSTM fold 4: (0.9270197844803011, 0.9274920084697832, 0.927018711339476, 0.9271412449783565)
GRU  fold 4: (0.9300986373225384, 0.9305988259812826, 0.9300981788984376, 0.9301911148578514)

===== FOLD 5 =====
LSTM fold 5: (0.9235988368778152, 0.9239164360343798, 0.9236014900613605, 0.9234763538064468)
GRU  fold 5: (0.9284451793146702, 0.9294247386614531, 0.92844

### Step 6 — Rata-rata skor

In [11]:
import numpy as np

print("\n===== Rata-rata Hasil K-Fold =====")

lstm_mean = np.mean(lstm_scores, axis=0)
gru_mean  = np.mean(gru_scores, axis=0)

print("LSTM (acc, prec, rec, f1):", lstm_mean)
print("GRU  (acc, prec, rec, f1):", gru_mean)


===== Rata-rata Hasil K-Fold =====
LSTM (acc, prec, rec, f1): [0.92641623 0.92697838 0.92641689 0.92644431]
GRU  (acc, prec, rec, f1): [0.92795568 0.92850259 0.92795644 0.92798959]


### Step 7 — Simpan model terlatih (F1 tertinggi)
Biar nanti bisa dipakai di fase inference.

In [12]:
best_model_name = "lstm" if lstm_mean[3] > gru_mean[3] else "gru"
config["best_model_type"] = best_model_name

with open("artifacts/config/config_aug.json", "w") as f:
    json.dump(config, f, indent=4)

print("Best model:", best_model_name)

os.makedirs("artifacts/model_final", exist_ok=True)

if best_model_name == "lstm":
    torch.save(lstm_model.state_dict(), "artifacts/model_final/final_model_aug.pth")
else:
    torch.save(gru_model.state_dict(), "artifacts/model_final/final_model.pth")

Best model: gru
